In [1]:
import pandas as pd
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
MODEL_PATH = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"  # deepseek-ai/DeepSeek-R1-Distill-Qwen-32B    meta-llama/Llama-3.1-8B-Instruct
TP_SIZE = 1  # 你的GPU数量

print(f" 正在加载 Tokenizer 和模型 {MODEL_PATH} ... ")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

# 初始化 vLLM 引擎
llm = LLM(
    model=MODEL_PATH,
    tensor_parallel_size=TP_SIZE,
    dtype="bfloat16",
    max_model_len=2048,            # 上下文截断
    gpu_memory_utilization=0.85,  # 显存预占比例
    enforce_eager=True            # 关闭 CUDA Graph 防止 OOM
)

 正在加载 Tokenizer 和模型 deepseek-ai/DeepSeek-R1-Distill-Llama-8B ... 
INFO 05-06 18:07:24 [utils.py:233] non-default args: {'dtype': 'bfloat16', 'max_model_len': 2048, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'deepseek-ai/DeepSeek-R1-Distill-Llama-8B'}
INFO 05-06 18:07:25 [model.py:549] Resolved architecture: LlamaForCausalLM
INFO 05-06 18:07:25 [model.py:1678] Using max model len 2048
INFO 05-06 18:07:25 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-06 18:07:25 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 05-06 18:07:25 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 05-06 18:07:25 [vllm.py:859] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 05-06 18:07:26 [vllm.py:1025] Cudagraph is disab

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


(EngineCore pid=869706) INFO 05-06 18:07:32 [weight_utils.py:825] Prefetching checkpoint files: 10% (1/2)
(EngineCore pid=869706) INFO 05-06 18:07:32 [weight_utils.py:825] Prefetching checkpoint files: 20% (2/2)
(EngineCore pid=869706) INFO 05-06 18:07:32 [weight_utils.py:843] Prefetching checkpoint files into page cache finished in 1.09s
(EngineCore pid=869706) INFO 05-06 18:07:33 [default_loader.py:384] Loading weights took 2.26 seconds
(EngineCore pid=869706) INFO 05-06 18:07:34 [gpu_model_runner.py:4820] Model loading took 14.99 GiB memory and 3.869843 seconds
(EngineCore pid=869706) INFO 05-06 18:07:36 [gpu_worker.py:436] Available KV cache memory: 21.85 GiB
(EngineCore pid=869706) INFO 05-06 18:07:36 [kv_cache_utils.py:1319] GPU KV cache size: 178,976 tokens
(EngineCore pid=869706) INFO 05-06 18:07:36 [kv_cache_utils.py:1324] Maximum concurrency for 2,048 tokens per request: 87.39x
(EngineCore pid=869706) INFO 05-06 18:07:36 [core.py:283] init engine (profile, create kv cache, wa

In [3]:
sampling_params = SamplingParams(
    temperature=0., 
    max_tokens=400, 
    # top_p=0.95,
    # stop=["\n\n", "User:"]
)

In [4]:
# 1. 手动挑选 10 个具有代表性的例子（包含正确答案的 Ground Truth 用于自动算分）
test_cases = [
    {"Sub": "car", "Rel": "on", "Obj": "street", "Expected": "Yes"},

    {"Sub": "person", "Rel": "wearing", "Obj": "hat", "Expected": "Yes"},
    {"Sub": "boat", "Rel": "flying in", "Obj": "room", "Expected": "No"},
    {"Sub": "cat", "Rel": "sitting on", "Obj": "table", "Expected": "Yes"},
    {"Sub": "pizza", "Rel": "eating", "Obj": "man", "Expected": "No"},
    {"Sub": "train", "Rel": "parked on", "Obj": "track", "Expected": "Yes"},
    {"Sub": "tree", "Rel": "riding", "Obj": "bike", "Expected": "No"},
    {"Sub": "dog", "Rel": "holding", "Obj": "branch", "Expected": "Yes"},
    {"Sub": "flower", "Rel": "growing on", "Obj": "rock", "Expected": "Yes"},

    {"Sub": "logo", "Rel": "of", "Obj": "banana", "Expected": "Yes"},

    {"Sub": "person", "Rel": "along", "Obj": "towel", "Expected": "No"},
    {"Sub": "vase", "Rel": "sitting on", "Obj": "airplane", "Expected": "No"},
    {"Sub": "bag", "Rel": "across", "Obj": "street", "Expected": "No"},
    {"Sub": "leg", "Rel": "of", "Obj": "branch", "Expected": "No"},
    {"Sub": "laptop", "Rel": "of", "Obj": "screen", "Expected": "No"},
    {"Sub": "girl", "Rel": "attached to", "Obj": "book", "Expected": "No"},
    {"Sub": "man", "Rel": "eating", "Obj": "surfboard", "Expected": "No"},
    {"Sub": "man", "Rel": "says", "Obj": "snowboard", "Expected": "No"},


    # --- 陷阱 1：主客体颠倒 / 逆向逻辑 (测试模型是否只看词语相关性，不看语序) ---
    {"Sub": "plant", "Rel": "eating", "Obj": "cow", "Expected": "No"},       # 牛吃草，草不能吃牛
    {"Sub": "car", "Rel": "riding", "Obj": "boy", "Expected": "No"},         # 男孩骑车/乘车，车不能骑男孩
    {"Sub": "food", "Rel": "eating", "Obj": "girl", "Expected": "No"},       # 食物不能吃人
    {"Sub": "number", "Rel": "says", "Obj": "sign", "Expected": "No"},       # 标牌上写着数字，数字不能"写着"标牌

    # --- 陷阱 2：物理尺度与承重常识 (测试模型是否考虑物体的实际大小和重量) ---
    {"Sub": "elephant", "Rel": "sitting on", "Obj": "chair", "Expected": "No"},  # 椅子承受不了大象
    {"Sub": "bird", "Rel": "carrying", "Obj": "horse", "Expected": "No"},        # 鸟叼不动马
    {"Sub": "cup", "Rel": "holding", "Obj": "elephant", "Expected": "No"},       # 杯子装不下大象
    {"Sub": "airplane", "Rel": "parked on", "Obj": "roof", "Expected": "No"},    # 真实世界的飞机不会停在普通屋顶上
    {"Sub": "car", "Rel": "in", "Obj": "box", "Expected": "No"},                 # 排除玩具特例，真车装不进普通纸箱

    # --- 陷阱 3：过度拟人化 / 动画片隐喻 (测试模型是否把童话当成现实) ---
    {"Sub": "motorcycle", "Rel": "wearing", "Obj": "helmet", "Expected": "No"},  # 摩托车不戴头盔，骑手才戴
    {"Sub": "shoe", "Rel": "walking on", "Obj": "sidewalk", "Expected": "No"},   # 鞋子本身不会走，必须是人穿着
    {"Sub": "dog", "Rel": "using", "Obj": "laptop", "Expected": "No"},           # 狗不具备使用电脑的物理能力
    {"Sub": "cat", "Rel": "wearing", "Obj": "boot", "Expected": "No"},           # 穿靴子的猫是童话，现实中视觉不合理
    {"Sub": "flower", "Rel": "holding", "Obj": "cup", "Expected": "No"},         # 花没有手，不能拿杯子
    {"Sub": "cabinet", "Rel": "wearing", "Obj": "coat", "Expected": "No"},       # 柜子不能“穿”外套（只能是里面挂着/装有）
    {"Sub": "bench", "Rel": "watching", "Obj": "bird", "Expected": "No"},        # 长椅没有视觉，不能 watching

    # --- 陷阱 4：生物特征错乱 / 机器与生物乱接 (测试解剖学常识) ---
    {"Sub": "wing", "Rel": "part of", "Obj": "dog", "Expected": "No"},           # 狗没有翅膀
    {"Sub": "leaf", "Rel": "part of", "Obj": "bird", "Expected": "No"},          # 鸟不长叶子
    {"Sub": "leg", "Rel": "attached to", "Obj": "wheel", "Expected": "No"},      # 真实的肉腿不会长在轮子上
    {"Sub": "wheel", "Rel": "belonging to", "Obj": "horse", "Expected": "No"},   # 马没有轮子
    {"Sub": "cow", "Rel": "sitting on", "Obj": "fence", "Expected": "No"},       # 牛的骨骼结构无法像人一样“坐”在篱笆上

    # --- 陷阱 5：空间介词混淆及极端特例 (讨论过的刁钻案例) ---
    {"Sub": "train", "Rel": "parked on", "Obj": "street", "Expected": "No"},     # 火车停在铁轨上，停在街上是极端特例
    {"Sub": "boat", "Rel": "parked on", "Obj": "tree", "Expected": "No"},        # 船不会停在树上
    {"Sub": "umbrella", "Rel": "flying in", "Obj": "room", "Expected": "No"},    # 雨伞在室内飞行（魔法/风暴极端情况）
    {"Sub": "logo", "Rel": "painted on", "Obj": "horse", "Expected": "No"},      # 活体马身上直接画Logo属于极端特例

    # --- 迷惑组：看似荒谬但物理现实完全成立的 Hard Positives (测试模型是否敢说 Yes) ---
    {"Sub": "bird", "Rel": "standing on", "Obj": "cow", "Expected": "Yes"},      # 牛背鹭等鸟类停在牛背上很常见
    {"Sub": "plant", "Rel": "growing on", "Obj": "building", "Expected": "Yes"}, # 爬山虎、苔藓长在建筑上
    {"Sub": "sign", "Rel": "says", "Obj": "number", "Expected": "Yes"},          # 标牌上写着限速等数字
    {"Sub": "cat", "Rel": "looking at", "Obj": "book", "Expected": "Yes"},       # 猫虽然不认字，但物理上盯着书看完全可能
    {"Sub": "dog", "Rel": "covered in", "Obj": "snow", "Expected": "Yes"},       # 狗在雪地玩耍身上覆满雪
    {"Sub": "tree", "Rel": "growing on", "Obj": "rock", "Expected": "Yes"},      # 树从岩石缝隙中长出来
    {"Sub": "bag", "Rel": "hanging from", "Obj": "branch", "Expected": "Yes"}    # 包挂在树枝上，视觉和物理上完全说得通
]

# 2. 匹配你原代码中的问题生成逻辑
def format_question(sub, rel, obj):
    sub_f = f'a group of {sub}' if sub in ['men', 'people'] else f'a {sub}'
    obj_f = f'a group of {obj}' if obj in ['men', 'people'] else f'a {obj}'
    
    rel_formatted = rel
    if rel != 'and':
        if rel == 'wears': rel_formatted = 'wearing'
        elif rel == 'says': rel_formatted = 'saying'
        elif rel == 'has': rel_formatted = 'have'
        
        rel_f = 'have' if rel_formatted == 'have' else f'be {rel_formatted}'
        return f"Can {sub_f} {rel_f} {obj_f}?"
    else:
        return f"Can {sub_f} {rel} {obj_f} appear together?"

# 3. 定义测试函数
def run_ablation(system_prompt_content, few_shots=[]):
    prompts = []
    
    # 构建所有问题的 Prompt
    for case in test_cases:
        question = format_question(case['Sub'], case['Rel'], case['Obj'])
        
        messages = [{"role": "system", "content": system_prompt_content}]
        messages.extend(few_shots)
        messages.append({"role": "user", "content": question})
        
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prompts.append(prompt_text)
        
    # vLLM 批量推理 (关闭 tqdm 防止 Jupyter 输出过长)
    outputs = llm.generate(prompts, sampling_params, use_tqdm=False)
    
    # 结果解析与统计
    results = []
    for case, out in zip(test_cases, outputs):
        answer = out.outputs[0].text.strip()
        # 简单判定：如果模型输出包含预期答案（忽略大小写）则判对
        is_correct = case['Expected'].lower() in answer.lower()
        results.append({
            "Question": format_question(case['Sub'], case['Rel'], case['Obj']),
            "Expected": case['Expected'],
            "Model_Output": answer,
            "Correct": is_correct
        })
        
    df_results = pd.DataFrame(results)
    accuracy = df_results['Correct'].mean() * 100
    print(f"\n=> 测试完成! 在 {len(df_results['Correct'])}个样本上的准确率为: {accuracy:.1f}%\n")
    return df_results

In [19]:
pd.set_option('display.max_colwidth', None)
prompt_v1 = """You are given a question in the form: 'Can a <Subject> be <Relationship> on a <Object>?'
Interpret this strictly as a statement about physical and commonsense plausibility in the real world.
Your task is to evaluate this question according to commonsense knowledge and visual plausibility.
For the given question, determine whether the situation is logically possible in the real world according to commonsense knowledge,
and whether this situation could realistically appear in a real photograph or video (i.e., no CGI, no fantasy, no metaphorical interpretations).
Ignore fictional, metaphorical, symbolic, microscopic, or extremely contrived edge cases unless explicitly stated.
Base your judgment on typical real-world conditions.
Respond with a Yes or No if both commonsense and visually plausible, without additional explainations."""

shots_v1 = [
    {"role": "user", "content": "Can a car be driving on a road?"},
    {"role": "assistant", "content": "Yes"},
    {"role": "user", "content": "Can an animal be eating an apple?"},
    {"role": "assistant", "content": "Yes"},
    {"role": "user", "content": "Can a bird be under a mountain?"},
    {"role": "assistant", "content": "No"},
    {"role": "user", "content": "Can an arm be flying in the sky?"},
    {"role": "assistant", "content": "No"}
]

print("=== 实验 1: 原始长 Prompt + 4-shot ===")
df_res1 = run_ablation(prompt_v1, shots_v1)
display(df_res1) # 

=== 实验 1: 原始长 Prompt + 4-shot ===

=> 测试完成! 在 50个样本上的准确率为: 86.0%



,Question,Expected,Model_Output,Correct
0,Can a car be on a street?,Yes,"Okay, so the user is asking if a car can be on a street. Hmm, that seems pretty straightforward. I mean, cars are designed to be on roads, right? But wait, I should think about this carefully. Is there any situation where a car wouldn't be on a street? Well, cars can be on streets, highways, maybe even on grass if it's a parking lot or a driveway. But the question is specifically about a street. Yeah, that makes sense. So, the answer should be yes. I don't see any reason why a car wouldn't be on a street. It's a common everyday scenario. I should just confirm that there's no trick here. The user might be testing if I can recognize something obvious, but it's better to be thorough. So, I'll say yes, a car can definitely be on a street.\n</think>\n\nYes. A car can be on a street.",True
1,Can a person be wearing a hat?,Yes,"Okay, so the user is asking if a person can be wearing a hat. Hmm, that seems pretty straightforward. I mean, I know people wear hats all the time. But wait, the user has a history of asking about more complex things, like cars on roads and animals eating apples. So maybe they're testing if I can handle even the most obvious questions.\n\nLet me think about the context. The previous questions were about physical possibilities and visual plausibility. The user wants a strict yes or no based on real-world knowledge, no metaphors or anything. So for this question, it's definitely yes. People wear hats daily, so it's both logically possible and visually plausible. No need for any complicated reasoning here. Just a simple answer should suffice.\n</think>\n\nYes.",True
2,Can a boat be flying in a room?,No,"Okay, so I'm trying to figure out whether a boat can fly in a room. Hmm, that's an interesting question. First, I know that boats are typically found on water, like rivers, lakes, or oceans. They float because they're designed to displace water and have a buoyant force. But a room is an enclosed space, usually indoors, where you live or work. So, can a boat actually be in a room?\n\nWell, I'm picturing a room, maybe like my living room or a bedroom. It's not near any water, right? So, how would a boat get there? Maybe if someone tried to put a small boat in their room, but that seems tricky. Boats are usually big and have wheels or other features that make them suitable for water. In a room, there's no water, so the boat wouldn't have anything to float on. It would just sit on the floor, but then it wouldn't be flying.\n\nWait, the question is about a boat flying in a room, not just being there. Flying implies moving through the air. So, can a boat actually take off and fly? That seems even more impossible because boats are not designed to generate lift. They rely on buoyancy and maybe a motor for movement. Flying would require some kind of lift, like wings or a helicopter, which a boat doesn't have.\n\nMaybe if someone tried to attach something to a boat to make it fly, but that's not really flying in the traditional sense. It would just be like a boat with some extra propellers or something, but that's not the same as flying. Plus, in a room, there's not enough space for a boat to generate lift. The walls and the ceiling are all around, so it's not like a boat could take off into the sky.\n\nAlso, from a commonsense perspective, boats are heavy and not meant to be in the air. They're built for water, not",True
3,Can a cat be sitting on a table?,Yes,"Okay, so I need to figure out if a cat can sit on a table. Well, cats are animals, and I know they can climb and jump. I've seen cats on tables before, like when they're trying to get to a higher place or when they're just lounging around. But wait, can they actually sit on a table without falling off? I mean, tables are flat and not that high, but cats have pretty good balance. Maybe they can, but I'm not sure if it's common or if it's something only some cats do.\n\nAlso, I wonder if the table

In [6]:
# 1. 重新配置采样参数（增加长度，temperature可以为0确保单次推理最稳）
sampling_params_cot = SamplingParams(temperature=0, max_tokens=1024)
pd.set_option('display.max_colwidth', None)
# 2. 定义 CoT Prompt
prompt_cot = """You are a commonsense reasoning assistant. 
For each question, analyze the physical and visual plausibility step by step.
Consider if the interaction is possible in the real world and could appear in a real photo.
Strictly evaluate the spatial relationship defined by the preposition. Do not substitute the relationship with a more plausible one (e.g., do not treat 'on' as 'in', or 'along' as 'with').
Do not invent stories or specific contexts to justify the situation. If it requires a 'dropped' or 'accident' scenario to be possible, respond with No.
Focus on the physical geometry. Can a human body physically align 'along' a towel's length? If the physical orientation is nonsensical, respond with No.


End your response with 'Final Answer: Yes' or 'Final Answer: No'."""

shots_cot = [
    {"role": "user", "content": "Can an arm be flying in the sky?"},
    {"role": "assistant", "content": "Reasoning: 1) An arm is a biological part of a human body. 2) In the real world, arms cannot detach and fly independently. 3) This would only happen in CGI or fantasy. \nFinal Answer: No"},
    {"role": "user", "content": "Can a car be driving on a road?"},
    {"role": "assistant", "content": "Reasoning: 1) Cars are vehicles designed for roads. 2) This is a standard real-world activity. 3) It is visually plausible. \nFinal Answer: Yes"},
    {"role": "user", "content": "Can a house be in a car?"},
    {"role": "assistant", "content": "Reasoning: 1) A house is significantly larger than a car. 2) A car cannot physically contain a house. 3) This violates physical scale and spatial logic. \nFinal Answer: No"},

]

# 3. 运行测试（复用你的 run_ablation 函数，但传入新的采样参数）
# 注意：原 run_ablation 内部使用了 sampling_params，我们需要临时替换它
import copy
original_params = sampling_params
sampling_params = sampling_params_cot

print("=== 实验 2: Chain-of-Thought (CoT) ===")
df_res2 = run_ablation(prompt_cot, shots_cot)
display(df_res2)

# 还原参数
sampling_params = original_params

=== 实验 2: Chain-of-Thought (CoT) ===

=> 测试完成! 在 50个样本上的准确率为: 92.0%



,Question,Expected,Model_Output,Correct
0,Can a car be on a street?,Yes,"Okay, so the user is asking if a car can be on a street. Hmm, that seems pretty straightforward. I mean, cars are designed to be on roads, right? They have wheels, they're built for driving on asphalt or concrete surfaces. But wait, the user's previous questions were a bit more abstract, like whether an arm can fly or a house can be in a car. Those were more about whether something impossible could happen. \n\nNow, this question is different. It's about a normal, everyday scenario. So, I need to think about the physical and visual plausibility. The user mentioned focusing on the spatial relationship defined by the preposition. In this case, ""on"" refers to the car being placed on the street. \n\nIs there any reason why a car couldn't be on a street? Well, cars are meant to be on streets. They're built to drive on them, so it's a common sight. There's no contradiction here. The user also said not to substitute the preposition, so I shouldn't think of ""on"" as ""in"" or anything else. \n\nI should also consider if there's any scenario where a car isn't on a street. Like, maybe in a parking lot or driveway, but the question specifically says ""street."" So, it's definitely possible. \n\nI don't think I need to invent any stories or contexts here. It's a straightforward yes. The user probably wants a clear, concise answer without any complications. They might be testing the model's understanding of spatial relationships. \n\nSo, putting it all together, the car being on a street is not only physically possible but also something we see every day. No need for any complicated reasoning beyond that.\n</think>\n\nYes, a car can be on a street. Cars are designed to be operated on roads and streets. This is a common and visually plausible scenario in the real world. \n\nFinal Answer: Yes",True
1,Can a person be wearing a hat?,Yes,"Okay, so the user is asking if a person can be wearing a hat. Hmm, that seems pretty straightforward. I mean, wearing a hat is a common everyday thing, right? People do it all the time. But let me break it down to make sure I'm not missing anything.\n\nFirst, I need to consider the physical aspects. A hat is designed to be worn on the head. It's made of fabric or other materials and is shaped to cover the head. So, putting it on someone's head should be possible. The hat isn't too heavy or too large that it can't balance on a person's head. In fact, people wear hats for various reasons, like protection from the sun or style.\n\nNext, thinking about the spatial relationship here. The preposition is ""wearing,"" which means the hat is placed on the person's head. That makes sense because the hat is intended to be worn that way. There's no confusion between ""wearing"" and ""holding"" or ""placing on the table."" It's specifically about the hat being in contact with the head.\n\nI should also consider if there's any scenario where this wouldn't make sense. Like, could a hat be worn in zero gravity? Well, in that case, it might float around, but the question is about a regular setting, probably on Earth. So, yeah, it's definitely possible.\n\nI don't see any reason to doubt this. It's a normal, everyday action. No need to overcomplicate it. The user is probably just curious about the possibility, maybe for a creative project or something. So, the answer should be a simple yes.\n</think>\n\nYes, a person can wear a hat. A hat is designed to be placed on a person's head, and this is a common and physically plausible action in the real world. The spatial relationship defined by ""wearing"" is that the hat is in contact with the person's head. \n\nFinal Answer: Yes",True
2,Can a boat be flying in a room?,No,"Okay, so I'm trying to figure out if a boat can be flying in a room. Hmm, that's an interesting question. Let me break it down step by step.\n\nFirst, what do I know about boats? Boats are typically watercraft designed to float on water. Th

In [8]:
# 1. 重新配置采样参数（增加长度，temperature可以为0确保单次推理最稳）
sampling_params_cot = SamplingParams(temperature=0, max_tokens=1024)
pd.set_option('display.max_colwidth', None)
# 2. 定义 CoT Prompt
prompt_cot2 = """You are a Physical Logic Auditor. Task: Evaluate <Subject, Relationship, Object> for real-world plausibility.

[STRICT OPERATING RULES]:
1. PREPOSITION RIGOR: Evaluate the EXACT preposition. 'Along' is not 'With'; 'On' is not 'In'. If the geometry is awkward, it's a No.
2. NO JUSTIFICATION: Do not invent 'accidents', 'CGI', or 'extreme edge cases'. If it's not a common visual, it's a No.
3. NO FILLERS: Start directly with "Reasoning:". Do not say "Okay", "I will analyze", or "Let's think".
4. GEOMETRY FOCUS: Focus on scale, volume, and physical orientation.

[OUTPUT FORMAT]:
Reasoning:
- [Scale/Part]: <Size or hierarchy check>
- [Geometry]: <Physical alignment/spatial check>
- [Agency]: <Capability check>
Final Answer: Yes/No"""

shots_cot = [
    {
        "role": "user", 
        "content": "Can a person be along a towel?"
    },
    {
        "role": "assistant", 
        "content": "Reasoning:\n- [Geometry]: 'Along' implies parallel alignment of lengths. A human body typically lies 'on' a towel; 'along' suggests a nonsensical geometric orientation.\n- [Plausibility]: Not a standard visual or physical arrangement.\nFinal Answer: No"
    },
    {
        "role": "user", 
        "content": "Can a laptop be of a screen?"
    },
    {
        "role": "assistant", 
        "content": "Reasoning:\n- [Scale/Part]: A screen is a component of a laptop. The relationship 'of' reverses the natural part-whole hierarchy.\n- [Logic]: A whole cannot be 'of' its part.\nFinal Answer: No"
    },
    {
        "role": "user", 
        "content": "Can a bag be across a street?"
    },
    {
        "role": "assistant", 
        "content": "Reasoning:\n- [Scale]: A street is significantly wider than a standard bag. A bag cannot physically span the width of a street.\n- [Geometry]: Dimension mismatch for the relationship 'across'.\nFinal Answer: No"
    },
    {
        "role": "user", 
        "content": "Can a car be driving on a road?"
    },
    {
        "role": "assistant", 
        "content": "Reasoning:\n- [Scale]: Sizes are compatible.\n- [Agency]: Cars are designed for this specific interaction and environment.\nFinal Answer: Yes"
    }
]
# 3. 运行测试（复用你的 run_ablation 函数，但传入新的采样参数）
# 注意：原 run_ablation 内部使用了 sampling_params，我们需要临时替换它
import copy
original_params = sampling_params
sampling_params = sampling_params_cot

print("=== 实验 2: Chain-of-Thought (CoT) ===")
df_res2 = run_ablation(prompt_cot2, [])
display(df_res2)

# 还原参数
sampling_params = original_params

=== 实验 2: Chain-of-Thought (CoT) ===

=> 测试完成! 在 50个样本上的准确率为: 90.0%



,Question,Expected,Model_Output,Correct
0,Can a car be on a street?,Yes,"Okay, so I need to evaluate whether the statement ""Can a car be on a street?"" is plausible based on the given rules. Let me break it down step by step.\n\nFirst, I'll look at the subject, which is the car. The relationship is ""be on,"" and the object is the street. The question is about the physical possibility of a car existing on a street.\n\nStarting with the preposition: ""on"" is correct here. It's not ""along"" or ""in,"" so that's good. The preposition makes sense because a car is placed on a street, not inside it or alongside it.\n\nNext, I need to consider the geometry. A car is a three-dimensional object, and a street is a two-dimensional surface. When a car is on a street, it's placed on the surface, so the orientation is correct. The car doesn't float or hover, so the placement is realistic. The scale is also appropriate; a car is the right size to be on a street.\n\nLooking at agency, cars are designed to be on streets. They have wheels and are built to move on surfaces like roads. So, a car has the capability to be on a street without any issues.\n\nI don't see any need to invent accidents or extreme cases here. Cars are commonly found on streets every day, so this is a normal scenario. There's no need for special effects or extreme conditions.\n\nPutting it all together, the statement is plausible because the preposition is correct, the geometry makes sense, and the car has the capability to be on a street. So, the answer should be yes.\n</think>\n\nReasoning:\n- [Scale/Part]: The car is a typical vehicle, and the street is a common place for it.\n- [Geometry]: The car is placed on the street, which is a flat, two-dimensional surface.\n- [Agency]: Cars are designed to operate on streets, making this placement feasible.\n\nFinal Answer: Yes",True
1,Can a person be wearing a hat?,Yes,"Okay, so I'm trying to figure out if a person can be wearing a hat. Let me break this down step by step. First, I need to understand the question properly. The user is asking if a person can wear a hat, so I need to assess the plausibility based on the given rules.\n\nStarting with the first rule, PREPOSITION RIGOR. The user wrote ""wearing a hat."" The preposition here is ""a."" I think that's correct because ""wearing a hat"" is a common phrase. If it were ""wearing with a hat"" or ""wearing on a hat,"" that might be incorrect, but ""a"" works here. So, the preposition is correct.\n\nNext, NO JUSTIFICATION. The user didn't add any extra details like accidents, CGI, or extreme cases. They just asked a straightforward question. Since it's a common scenario, I don't need to invent anything. It's a normal situation, so that's fine.\n\nNO FILLERS. The response should start with ""Reasoning:"" and not with any other phrases like ""Okay"" or ""I will analyze."" I need to make sure to follow that structure.\n\nGEOMETRY FOCUS. I should consider the scale, volume, and spatial orientation. A hat is designed to be worn on the head, which is a specific part of the body. It doesn't require any extreme scaling or unusual placement. The hat sits on top of the head without affecting the body's structure. So, the geometry makes sense.\n\nAGENCY. People have the capability to wear hats as clothing, so their agency is relevant here. It's a common human action.\n\nPutting it all together, the answer should be yes because the preposition is correct, there's no need for justification, the geometry is sound, and it's a normal human action.\n</think>\n\nReasoning:\n- [Scale/Part]: The hat is a small object designed to be worn on the head, which is a specific part of the body. It doesn't require extreme scaling or unusual placement.\n- [Geometry]: The hat sits on top of the head without interfering with the body's structure, making it a plausible spatial arrangement.\n- [Agency]: People have the capability to wear hats as part of their clothing, making it a common and feasible actio

In [ ]:
# 1. 重新配置采样参数（增加长度，temperature可以为0确保单次推理最稳）
sampling_params_cot = SamplingParams(temperature=0, max_tokens=512)
pd.set_option('display.max_colwidth', None)
# 2. 定义 CoT Prompt
prompt_cot = """You are a Physical Logic Auditor. Task: Evaluate <Subject, Relationship, Object> for real-world plausibility.

[STRICT OPERATING RULES]:
1. PREPOSITION RIGOR: Evaluate the EXACT preposition. 'Along' is not 'With'; 'On' is not 'In'. If the geometry is awkward, it's a No.
2. NO JUSTIFICATION: Do not invent 'accidents', 'CGI', or 'extreme edge cases'. If it's not a common visual, it's a No.
3. NO FILLERS: Start directly with "Reasoning:". Do not say "Okay", "I will analyze", or "Let's think".
4. GEOMETRY FOCUS: Focus on scale, volume, and physical orientation.

[OUTPUT FORMAT]:
Reasoning:
- [Scale/Part]: <Size or hierarchy check>
- [Geometry]: <Physical alignment/spatial check>
- [Agency]: <Capability check>
Final Answer: Yes/No"""

shots_cot = [
    {"role": "user", "content": "Can an arm be flying in the sky?"},
    {"role": "assistant", "content": "Reasoning: 1) An arm is a biological part of a human body. 2) In the real world, arms cannot detach and fly independently. 3) This would only happen in CGI or fantasy. \nFinal Answer: No"},
    {"role": "user", "content": "Can a car be driving on a road?"},
    {"role": "assistant", "content": "Reasoning: 1) Cars are vehicles designed for roads. 2) This is a standard real-world activity. 3) It is visually plausible. \nFinal Answer: Yes"},
    {"role": "user", "content": "Can a house be in a car?"},
    {"role": "assistant", "content": "Reasoning: 1) A house is significantly larger than a car. 2) A car cannot physically contain a house. 3) This violates physical scale and spatial logic. \nFinal Answer: No"},

]

# 3. 运行测试（复用你的 run_ablation 函数，但传入新的采样参数）
# 注意：原 run_ablation 内部使用了 sampling_params，我们需要临时替换它
import copy
original_params = sampling_params
sampling_params = sampling_params_cot

print("=== 实验 2: Chain-of-Thought (CoT) ===")
df_res2 = run_ablation(prompt_cot, shots_cot)
display(df_res2)

# 还原参数
sampling_params = original_params